## 📖 Dicionário de Dados

| **Coluna** | **Descrição** |
|---|---|
| `DATA` | Data da compra |
| `CO_ID` | Identificação da compra (número da nota fiscal) |
| `CL_ID` | Identificação do cliente (número do cliente) |
| `CL_GENERO` | Sexo biológico informado pelo cliente |
| `CL_EC` | Estado civil do cliente |
| `CL_FHL` | Número de filhos do cliente |
| `CL_SEG` | Segmentação econômica do cliente (Classe A, B ou C) |
| `PR_ID` | Código do produto adquirido (SKU) |
| `PR_CAT` | Categoria do produto adquirido |
| `PR_NOME` | Nome do produto adquirido |

### Valores de `CL_EC` (Estado Civil)

| **Código** | **Descrição** |
|---|---|
| `1` | Casado ou união estável |
| `2` | Divorciado |
| `3` | Separado |
| `4` | Solteiro |
| `5` | Viúvo |

## 🔍 Design dos dados importados

| **Critério** | **DATA** | **CO_ID** | **CL_ID** | **CL_GENERO** | **CL_EC** | **CL_FHL** | **CL_SEG** | **PR_ID** | **PR_CAT** | **PR_NOME** |
|---|---|---|---|---|---|---|---|---|---|---|
| Tipo de dado atual | str | int64 | int64 | str | int64 | int64 | str | int64 | str | str |
| Tipo de dado após tratamento | datetime64 | int64 | int64 | string | string | int64 | string | int64 | string | string |

## 
| **Critério** | **Resultado** |
|---|---|
| Duplicatas | 96553 Linhas |
| Valores nulos | Encontrados em algumas colunas |
| Impacto no negócio | Pode comprometer análises e segmentações |

## ETAPA DE IMPORTAÇÃO DE BIBLIOTECAS QUE SERÃO USADAS

In [355]:
import sys
import numpy as np
import pandas as pd
import importlib
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import utils.data_utils as du

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)


## IMPORTANDO OS DADOS DO CSV

In [356]:
caminho = "../data/base_varejo.csv"
df = pd.read_csv (caminho,sep=";")

## Visualizações iniciais do DF criado ##

In [357]:
# Primeiras 5 linhas , para confirmar a leitura do arquivo
print (df.head(5))

         DATA  CO_ID  CL_ID CL_GENERO  CL_EC  CL_FHL CL_SEG  PR_ID     PR_CAT  \
0  01/02/2019   1000    534         M      4       1      C     67    BEBIDAS   
1  01/02/2019   1000    534         M      4       1      C     70    BEBIDAS   
2  01/02/2019   1000    534         M      4       1      C    178    HIGIENE   
3  01/02/2019   1000    534         M      4       1      C      4  ALIMENTOS   
4  01/02/2019   1000    534         M      4       1      C    175    LIMPEZA   

                PR_NOME  Unnamed: 10  Unnamed: 11  Unnamed: 12  Unnamed: 13  
0  REFRIGERANTE GUARANA          NaN          NaN          NaN          NaN  
1   REFRIGERANTE OUTROS          NaN          NaN          NaN          NaN  
2       LENCO UMEDECIDO          NaN          NaN          NaN          NaN  
3               ABACAXI          NaN          NaN          NaN          NaN  
4     LIMPADOR MULTIUSO          NaN          NaN          NaN          NaN  


In [358]:
# Tipo dos dados encontrados
print (df.dtypes)

DATA               str
CO_ID            int64
CL_ID            int64
CL_GENERO          str
CL_EC            int64
CL_FHL           int64
CL_SEG             str
PR_ID            int64
PR_CAT             str
PR_NOME            str
Unnamed: 10    float64
Unnamed: 11    float64
Unnamed: 12    float64
Unnamed: 13    float64
dtype: object


In [359]:
# Linhas X Colunas
print (f"Quantidade de Linhas e Colunas localizadas")
print (f"Linhas: {df.shape[0]} \nColunas: {df.shape[1]}")

Quantidade de Linhas e Colunas localizadas
Linhas: 830000 
Colunas: 14


In [360]:
print ("\nVisualização das estatísticas dos dados (Quartis/Media/Minimo/Max)\n")
print (df.describe())


Visualização das estatísticas dos dados (Quartis/Media/Minimo/Max)

          CO_ID     CL_ID     CL_EC    CL_FHL     PR_ID  Unnamed: 10  \
count 830000.00 830000.00 830000.00 830000.00 830000.00         0.00   
mean  460045.09    499.60      2.60      1.15    115.05          NaN   
std   265465.25    287.57      1.17      1.42     66.13          NaN   
min     1000.00      1.00      1.00      0.00      1.00          NaN   
25%   233117.00    254.00      2.00      0.00     58.00          NaN   
50%   456517.00    498.00      3.00      0.00    115.00          NaN   
75%   690132.00    746.00      4.00      2.00    172.00          NaN   
max   919822.00   1000.00      5.00      4.00    229.00          NaN   

       Unnamed: 11  Unnamed: 12  Unnamed: 13  
count         0.00         0.00         0.00  
mean           NaN          NaN          NaN  
std            NaN          NaN          NaN  
min            NaN          NaN          NaN  
25%            NaN          NaN          NaN  


In [361]:
print ("\nSomatória de valores nulos por Coluna, aqui verificamos a existência de 4 colunas não nomeadas que estão todas nulas")
print (df.isnull().sum())

print("\nColunas com nulos:")
print(df.isnull().any())


Somatória de valores nulos por Coluna, aqui verificamos a existência de 4 colunas não nomeadas que estão todas nulas
DATA                0
CO_ID               0
CL_ID               0
CL_GENERO           0
CL_EC               0
CL_FHL              0
CL_SEG              0
PR_ID               0
PR_CAT              0
PR_NOME             0
Unnamed: 10    830000
Unnamed: 11    830000
Unnamed: 12    830000
Unnamed: 13    830000
dtype: int64

Colunas com nulos:
DATA           False
CO_ID          False
CL_ID          False
CL_GENERO      False
CL_EC          False
CL_FHL         False
CL_SEG         False
PR_ID          False
PR_CAT         False
PR_NOME        False
Unnamed: 10     True
Unnamed: 11     True
Unnamed: 12     True
Unnamed: 13     True
dtype: bool


## REALIZADO UMA PRIMEIRA INSPEÇÃO NO DOCUMENTO IMPORTADO, INICIA AGORA OS TRATAMENTOS DOS DADOS

1ª REMOÇÃO DAS COLUNAS (UNNAMED: 10 / UNNAMED: 11 / UNNAMED: 12 / UNNAMED: 13)

2ª SUBSTITUIÇÃO DE STRINGS VAZIAS E NULLS UTILIZANDO O NP.NAN PARA OBTER NAN REAL

3ª PADRONIZAR GÊNERO F/M PARA FEMININO/MASCULINO

4ª PADRONIZAR ESTADO CIVIL DE INT (1/2/3/4/5) PARA O CORRESPONDENTE EM STR (CASADO OU UNIÃO ESTÁVEL, DIVORCIADO, SEPARADO, SOLTEIRO, VIÚVO)

5ª TRATAR CATEGORIAS SEM INFORMAÇÃO COM O VALOR "SEM CATEGORIA"

6ª TRATAR DATAS ANTES EM STRING E TRANSFORMÁ-LAS EM DATETIME UTILIZANDO O FORMATO BRASILEIRO

In [362]:
# DROP Colunas zeradas
df = df.drop(columns=['Unnamed: 10', 'Unnamed: 11','Unnamed: 12','Unnamed: 13'])
# Confirmação que as colunas foram excluídas:
print (df.shape)
# Antes da remoção, eram 830000 x 14, confirmando assim a exclusão

(830000, 10)


In [363]:
# Substituição de String Vazias e Nulls utilizando o np.nan para obter NaN real

df = df.replace(['NULL', 'N/A', '', '#N/D',  ' '], np.nan, inplace=True)
print(df.isnull().sum())

DATA            0
CO_ID           0
CL_ID           0
CL_GENERO       0
CL_EC           0
CL_FHL          0
CL_SEG          0
PR_ID           0
PR_CAT       3650
PR_NOME      3650
dtype: int64


In [364]:
# Verificação de Duplicatas e dropando elas

# Para fins de backup, guardo as informações que foram localizadas num novo DF , e depois dropo do DF principal
total_duplicatas = df.duplicated().sum()

print (df[df.duplicated()])
print(f"Antes: {df.shape[0]} linhas")

print ("".center(100,'='))

df = df.drop_duplicates().reset_index(drop=True)
print(f"Depois: {df.shape[0]} linhas")



              DATA   CO_ID  CL_ID CL_GENERO  CL_EC  CL_FHL CL_SEG  PR_ID  \
19      01/02/2019    1000    534         M      4       1      C     13   
40      01/02/2019    1000    534         M      4       1      C      4   
46      01/02/2019    1000    534         M      4       1      C     11   
49      01/02/2019    1000    534         M      4       1      C     69   
50      01/02/2019    1000    534         M      4       1      C    218   
...            ...     ...    ...       ...    ...     ...    ...    ...   
829989  19/08/2022  919822    155         F      2       0      B     34   
829992  19/08/2022  919822    155         F      2       0      B     30   
829995  19/08/2022  919822    155         F      2       0      B    183   
829996  19/08/2022  919822    155         F      2       0      B     56   
829997  19/08/2022  919822    155         F      2       0      B    227   

           PR_CAT             PR_NOME  
19      ALIMENTOS              BANANA  
40     

In [365]:
# Padronizar Gênero F/M por FEMINIMO/MASCULINO
print (f"Antes da padronizaçao") 
print (df['CL_GENERO'].head(5))

df['CL_GENERO'] = du.padroniza_genero(df)
print (f"\nApós padronizaçao") 
print (df['CL_GENERO'].head(5))

Antes da padronizaçao
0    M
1    M
2    M
3    M
4    M
Name: CL_GENERO, dtype: str

Após padronizaçao
0    MASCULINO
1    MASCULINO
2    MASCULINO
3    MASCULINO
4    MASCULINO
Name: CL_GENERO, dtype: str


In [366]:
# Padronizar Estado Civil de INT (1/2/3/4/5) para o correspodente em STR (CASADO OU UNIÃO ESTÁVEL, DIVORCIADO, SEPARADO, SOLTEIRO, VIÚVO)

print (f"Antes da padronizaçao")
print (df['CL_EC'].head(5)) 

df['CL_EC'] = du.padroniza_estado_civil(df)
print (f"\nApós padronizaçao") 
print (df['CL_EC'].head(5)) 

Antes da padronizaçao
0    4
1    4
2    4
3    4
4    4
Name: CL_EC, dtype: int64

Após padronizaçao
0    SOLTEIRO
1    SOLTEIRO
2    SOLTEIRO
3    SOLTEIRO
4    SOLTEIRO
Name: CL_EC, dtype: str


In [367]:
# TRATAR CATEGORIAS ONDE NÃO HÁ INFORMAÇÃO POR "SEM CATEGORIA"

print (f"Antes da padronizaçao")
print (df['PR_CAT'].unique().tolist()) 

df['PR_CAT'] = du.tratar_categorias(df)
print (f"Após padronizaçao")
print (df['PR_CAT'].unique().tolist()) 

Antes da padronizaçao
['BEBIDAS', 'HIGIENE', 'ALIMENTOS', 'LIMPEZA', 'ACESSORIOS', 'PET', nan]
Após padronizaçao
['BEBIDAS', 'HIGIENE', 'ALIMENTOS', 'LIMPEZA', 'ACESSORIOS', 'PET', 'SEM CATEGORIA']


In [368]:
# TRATAR DATAS ANTES EM STRING E TRANSFORMÁ-LAS EM DATETIME USANDO O FORMATO BRASILEIRO
print (f"Antes dos tratamentos: ")
print('Datas inválidas (NaT):', df['DATA'].isna().sum())
print('Tipo da coluna data:', df['DATA'].dtype)
print ("".center(100,"*"))
df['DATA'] = du.tratar_datas(df)
print (f"Após tratamentos: ")
print('Datas inválidas (NaT):', df['DATA'].isna().sum())
print('Tipo da coluna data:', df['DATA'].dtype)

Antes dos tratamentos: 
Datas inválidas (NaT): 0
Tipo da coluna data: str
****************************************************************************************************
Após tratamentos: 
Datas inválidas (NaT): 0
Tipo da coluna data: datetime64[us]


# DICIONÁRIO DE DADOS

* qtd_prod_venda | Groupby de vendas usando a coluna PR_NOME
* qtd_cat_venda  | Groupby de categorias usando a coluna PR_CAT
* cli_freq_compras | Groupby de frequência de compras por cliente, usando CL_ID e CO_ID 
* perfil_compra_por_filho  | Perfil de categorias mais compradas dependendo da quantidade de filhos, usando CL_FHL e PR_CAT 
* vendas_mensais_por_cat | Verica por Mês as categorias mais vendidas, pegando os top 5 , da data mais recente até a última data


## ETAPA DAS ANAĹISES

In [369]:
# Groupby de vendas usando a coluna PR_NOME

qtd_prod_venda = (
    df.groupby('PR_NOME')
    .size()
    .reset_index(name='Quantidade')
    .sort_values('Quantidade',ascending=False)
    .reset_index(drop=True)
)
print ("Top 5 produtos mais vendidos")
print (qtd_prod_venda.head(5))
print (''.center(100,"="))
print ("Top 5 produtos menos vendidos")
print (qtd_prod_venda.tail(5))

Top 5 produtos mais vendidos
           PR_NOME  Quantidade
0  PRESUNTO COZIDO       12719
1         SARDINHA        6610
2           BANANA        6518
3  ESCOVA DE DENTE        6518
4              GEL        6517
Top 5 produtos menos vendidos
                   PR_NOME  Quantidade
112                ABACATE        3232
113             ABSORVENTE        3220
114  ALIMENTO PARA PASSARO        3198
115                   ALHO        3194
116                 ALCOOL        3168


In [370]:
# Categorias que mais vendem usando a coluna PR_CAT

qtd_cat_venda = (
    df.groupby('PR_CAT')
    .size()
    .reset_index(name='Quantidade')
    .sort_values('Quantidade',ascending=False)
    .reset_index(drop=True)
)

print ("Top 3 Categorias que mais vendem")
print (qtd_cat_venda.head(3))
print (''.center(100,"="))
print ("Top 3 Categorias que menos vendem")
print (qtd_cat_venda.tail(3))

Top 3 Categorias que mais vendem
      PR_CAT  Quantidade
0  ALIMENTOS      384197
1    HIGIENE      137702
2    LIMPEZA      128632
Top 3 Categorias que menos vendem
          PR_CAT  Quantidade
4            PET       28553
5     ACESSORIOS       12871
6  SEM CATEGORIA        3228


In [371]:
# CLIENTES COM MAIOR QUANTIDADE DE COMPRAS, USANDO AS COLUNAS DO CL_ID (CLIENTE ID) E CO_ID (NF), COMANDO NUNIQUE 

cli_freq_compras = (df.groupby('CL_ID')['CO_ID']
               .nunique()
               .reset_index(name='Qtd de Compras')
               .sort_values('Qtd de Compras',ascending=False)
               .reset_index(drop=True)
               )
print (f"Rank top 5 melhores clientes: ")
print (cli_freq_compras.head(5))
print (''.center(100,"="))
print (f"Rank top 5 clientes com menos compras: ")
print (cli_freq_compras.tail(5))

Rank top 5 melhores clientes: 
   CL_ID  Qtd de Compras
0     41              34
1    822              33
2    793              32
3    122              32
4    485              31
Rank top 5 clientes com menos compras: 
     CL_ID  Qtd de Compras
995    716               8
996    557               8
997    490               7
998     20               7
999    216               7


In [372]:

# Perfil de categorias mais compradas dependendo da quantidade de filhos, usando CL_FHL e PR_CAT 

perfil_compra_por_filho = (
    df.groupby(['CL_FHL', 'PR_CAT'])
    .size()
    .reset_index(name='Quantidade')
    .sort_values(['CL_FHL', 'Quantidade'],ascending=[False, False] )
    .groupby('CL_FHL')
    .head(5)
    .reset_index(drop=True)
)

print(perfil_compra_por_filho)

    CL_FHL     PR_CAT  Quantidade
0        4  ALIMENTOS       37361
1        4    HIGIENE       13248
2        4    LIMPEZA       12494
3        4    BEBIDAS        3665
4        4        PET        2724
5        3  ALIMENTOS       48284
6        3    HIGIENE       17487
7        3    LIMPEZA       16148
8        3    BEBIDAS        4812
9        3        PET        3587
10       2  ALIMENTOS       49269
11       2    HIGIENE       17647
12       2    LIMPEZA       16583
13       2    BEBIDAS        4895
14       2        PET        3721
15       1  ALIMENTOS       47613
16       1    HIGIENE       17018
17       1    LIMPEZA       15782
18       1    BEBIDAS        4901
19       1        PET        3561
20       0  ALIMENTOS      201670
21       0    HIGIENE       72302
22       0    LIMPEZA       67625
23       0    BEBIDAS       19991
24       0        PET       14960


In [373]:
#  Verica por Mês as categorias mais vendidas, pegando os top 5 , da data mais recente até a última data

vendas_mensais_por_cat = (
    df.groupby(['DATA', 'PR_CAT'])
    .size()
    .reset_index(name='VENDAS')
    .sort_values(['DATA', 'VENDAS'],ascending=[False, False] )
    .groupby('DATA')
    .head(5)
    .reset_index(drop=True)
)

print(vendas_mensais_por_cat)


           DATA     PR_CAT  VENDAS
0    2022-12-08  ALIMENTOS    2002
1    2022-12-08    HIGIENE     697
2    2022-12-08    LIMPEZA     648
3    2022-12-08    BEBIDAS     189
4    2022-12-08        PET     142
...         ...        ...     ...
1660 2019-01-04  ALIMENTOS     436
1661 2019-01-04    LIMPEZA     173
1662 2019-01-04    HIGIENE     151
1663 2019-01-04    BEBIDAS      40
1664 2019-01-04        PET      34

[1665 rows x 3 columns]
